## Dataset Mapping

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Yolo training

In [ ]:
# Install required packages if you don't have them
!pip install ultralytics huggingface_hub opencv-python numpy

import cv2
import numpy as np
from pathlib import Path
from typing import List, Dict
import json
import os
from huggingface_hub import hf_hub_download, list_repo_files
from ultralytics import YOLO
import random

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [ ]:
# Map COCO classes to our construction hazard classes
COCO_TO_HAZARD = {
    'person': 'person',
    'truck': 'dump_truck',
    'car': 'dump_truck',
    'bus': 'dump_truck',
}

# Construction Hazard Detection model classes
CONSTRUCTION_HAZARD_CLASSES = {
    'person': 'person',
    'machinery': 'excavator',
    'vehicle': 'dump_truck',
    'hardhat': 'hardhat',
    'mask': 'mask',
    'no-hardhat': 'no_hardhat',
    'no-mask': 'no_mask',
    'no-safety vest': 'no_safety_vest',
    'safety cone': 'safety_cone',
    'safety vest': 'safety_vest',
}

# Generic construction classes mapping
CONSTRUCTION_CLASSES = {
    'person': 'person',
    'excavator': 'excavator',
    'crane': 'crane',
    'ladder': 'ladder',
    'scaffold': 'scaffold',
    'dump_truck': 'dump_truck',
    'forklift': 'forklift',
    'truck': 'dump_truck',
    'car': 'dump_truck',
    'machinery': 'excavator',
    'vehicle': 'dump_truck',
}

# Hazard detection rules
HAZARD_RULES = {
    "suspended_load_hazard": {
        "description": "Suspended load hazard - crane/lifting equipment with person nearby",
        "requires": [["crane"], "person"],
        "severity": "high"
    },
    "heavy_equipment_proximity": {
        "description": "Heavy equipment proximity - person near heavy machinery",
        "requires": [["excavator", "dump_truck", "forklift"], "person"],
        "severity": "high"
    },
    "person_on_ladder": {
        "description": "Person on ladder - potential fall hazard",
        "requires": [["ladder"], "person"],
        "severity": "medium"
    },
    "person_on_scaffold": {
        "description": "Person on scaffold - height safety concern",
        "requires": [["scaffold"], "person"],
        "severity": "medium"
    },
    "ppe_violation_no_hardhat": {
        "description": "PPE violation - person without hardhat",
        "requires": [["no_hardhat"], "person"],
        "severity": "high"
    },
    "ppe_violation_no_safety_vest": {
        "description": "PPE violation - person without safety vest",
        "requires": ["no_safety_vest"],
        "severity": "high"
    },
    "ppe_violation_no_mask": {
        "description": "PPE violation - person without mask",
        "requires": [["no_mask"], "person"],
        "severity": "medium"
    }
}

In [ ]:
def calculate_proximity(bbox1: List[float], bbox2: List[float]) -> float:
    """Calculate distance between two bounding boxes (normalized)."""
    cx1 = (bbox1[0] + bbox1[2]) / 2
    cy1 = (bbox1[1] + bbox1[3]) / 2
    cx2 = (bbox2[0] + bbox2[2]) / 2
    cy2 = (bbox2[1] + bbox2[3]) / 2
    return np.sqrt((cx1 - cx2)**2 + (cy1 - cy2)**2)

def map_class_name(class_name: str, model_names: dict) -> str:
    """Map detected class to our hazard class."""
    class_name_lower = class_name.lower()

    if class_name_lower in CONSTRUCTION_HAZARD_CLASSES:
        return CONSTRUCTION_HAZARD_CLASSES[class_name_lower]

    # Handle variations
    if class_name_lower in ['hardhat', 'helmet']: return 'hardhat'
    if class_name_lower in ['no-hardhat', 'no hardhat', 'no_hardhat']: return 'no_hardhat'
    if class_name_lower in ['mask', 'face mask']: return 'mask'
    if class_name_lower in ['no-mask', 'no mask', 'no_mask']: return 'no_mask'
    if class_name_lower in ['safety vest', 'vest', 'safety_vest']: return 'safety_vest'
    if class_name_lower in ['no-safety vest', 'no safety vest', 'no_safety_vest', 'no-safety-vest']: return 'no_safety_vest'
    if class_name_lower in ['safety cone', 'cone', 'safety_cone']: return 'safety_cone'
    if class_name_lower in ['person', 'people', 'worker']: return 'person'
    if class_name_lower in ['machinery', 'machine', 'equipment', 'excavator', 'digger']: return 'excavator'
    if class_name_lower in ['vehicle', 'truck', 'car', 'dump truck', 'dump_truck']: return 'dump_truck'
    if class_name_lower in CONSTRUCTION_CLASSES: return CONSTRUCTION_CLASSES[class_name_lower]
    if class_name_lower in COCO_TO_HAZARD: return COCO_TO_HAZARD[class_name_lower]

    # Fuzzy matching fallbacks
    if 'crane' in class_name_lower or 'hook' in class_name_lower or 'load' in class_name_lower: return 'crane'
    if 'ladder' in class_name_lower: return 'ladder'
    if 'scaffold' in class_name_lower: return 'scaffold'
    if 'forklift' in class_name_lower or 'fork' in class_name_lower: return 'forklift'

    return None

def detect_hazards(detections: List[Dict], proximity_threshold: float = 0.3) -> List[Dict]:
    """Detect hazards based on object combinations and proximity."""
    hazards = []
    by_class = {}

    for det in detections:
        class_name = det["class_name"]
        if class_name not in by_class:
            by_class[class_name] = []
        by_class[class_name].append(det)

    for hazard_name, rule in HAZARD_RULES.items():
        required = rule["requires"]

        if len(required) == 2:
            req1, req2 = required
            req1_detections = []
            if isinstance(req1, list):
                for c_name in req1:
                    if c_name in by_class: req1_detections.extend(by_class[c_name])
            else:
                req1_detections = by_class.get(req1, [])

            req2_detections = by_class.get(req2, []) if isinstance(req2, str) else []

            if req1_detections and req2_detections:
                for det1 in req1_detections:
                    for det2 in req2_detections:
                        distance = calculate_proximity(det1["bbox"], det2["bbox"])
                        if distance < proximity_threshold:
                            hazards.append({
                                "hazard_type": hazard_name,
                                "description": rule["description"],
                                "severity": rule["severity"],
                                "objects": [
                                    {"class": det1["class_name"], "confidence": det1["confidence"]},
                                    {"class": det2["class_name"], "confidence": det2["confidence"]}
                                ],
                                "proximity": float(distance)
                            })

        elif len(required) == 1:
            req = required[0]
            if isinstance(req, list):
                for class_name in req:
                    if class_name in by_class and by_class[class_name]:
                        for det in by_class[class_name]:
                            hazards.append({
                                "hazard_type": hazard_name,
                                "description": rule["description"],
                                "severity": rule["severity"],
                                "objects": [{"class": det["class_name"], "confidence": det["confidence"]}]
                            })
                        break
            else:
                if req in by_class and by_class[req]:
                    for det in by_class[req]:
                        hazards.append({
                            "hazard_type": hazard_name,
                            "description": rule["description"],
                            "severity": rule["severity"],
                            "objects": [{"class": det["class_name"], "confidence": det["confidence"]}]
                        })
    return hazards

In [ ]:
def download_model(model_size="n"):
    """Download YOLO11 construction hazard detection model from HuggingFace."""
    repo_id = "yihong1120/Construction-Hazard-Detection-YOLO11"

    print("Checking available files in repository...")
    try:
        files = list_repo_files(repo_id=repo_id, repo_type="model")
        model_files = [f for f in files if f.endswith('.pt')]
    except Exception as e:
        print(f"Could not list files: {e}")
        model_files = []

    possible_paths = [
        f"models/pt/yolo11{model_size}.pt",
        f"yolo11{model_size}.pt",
        f"models/yolo11{model_size}.pt",
        f"best.pt",
    ]
    if model_files: possible_paths = model_files[:1] + possible_paths

    model_path = None
    for path in possible_paths:
        try:
            print(f"Trying to download: {path}")
            downloaded_path = hf_hub_download(repo_id=repo_id, repo_type="model", filename=path, local_dir="models")
            model_path = Path(downloaded_path)
            if model_path.exists():
                print(f"✅ Model downloaded successfully to {model_path.absolute()}")
                break
        except Exception as e:
            continue

    return str(model_path) if model_path else None

In [ ]:
def process_video_file(model_path, video_path, conf_threshold=0.25, output_dir="hazard_results"):
    """Run inference and hazard detection on a saved video file."""
    model = YOLO(model_path)
    model_names = model.names

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {video_path}")

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    video_name = Path(video_path).stem
    output_video_path = output_path / f"{video_name}_hazards.mp4"

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(str(output_video_path), fourcc, fps, (width, height))

    frame_count, total_detections, total_hazards = 0, 0, 0
    detected_classes = set()
    all_detections = []

    print(f"Processing {total_frames} frames...")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break

        results = model(frame, conf=conf_threshold, verbose=False)
        result = results[0]

        detections = []
        if result.boxes is not None:
            boxes = result.boxes
            for i in range(len(boxes)):
                bbox = boxes.xyxy[i].cpu().numpy()
                conf = float(boxes.conf[i].cpu().numpy())
                class_id = int(boxes.cls[i].cpu().numpy())
                detected_class = model_names[class_id]

                hazard_class = map_class_name(detected_class, model_names)
                if hazard_class:
                    detections.append({
                        "class_name": hazard_class, "original_class": detected_class,
                        "bbox": bbox.tolist(), "confidence": conf
                    })
                    detected_classes.add(hazard_class)
                    all_detections.append(hazard_class)

        hazards = detect_hazards(detections)
        total_detections += len(detections)
        total_hazards += len(hazards)

        annotated_frame = result.plot()

        if hazards:
            y_offset = 30
            for hazard in hazards[:3]:
                text = f"WARNING: {hazard['hazard_type']} ({hazard['severity']})"
                cv2.putText(annotated_frame, text, (10, y_offset), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
                y_offset += 30

        out.write(annotated_frame)
        frame_count += 1
        if frame_count % 30 == 0:
            print(f"Processed {frame_count}/{total_frames} frames... Detections: {total_detections}, Hazards: {total_hazards}")

    cap.release()
    out.release()
    print(f"Done! Video saved to {output_video_path}")

In [ ]:
# 1. Download the 'small' model (a good balance of speed/accuracy)
print("Step 1: Downloading weights...")
my_model_path = download_model(model_size="s")

# 2. Define your input video path
# UPDATE THIS string to point to a test video on your machine or in your colab environment
my_video_path = "/content/Test-Vids/ironsite-2.mp4"

# 3. Run the processing function
if my_model_path:
    if os.path.exists(my_video_path):
        print("\nStep 2: Starting Video Processing...")
        process_video_file(
            model_path=my_model_path,
            video_path=my_video_path,
            conf_threshold=0.30
        )
    else:
        print(f"Error: Could not find the video file at '{my_video_path}'. Please upload it or fix the path.")
else:
    print("Error: Model download failed.")

Step 1: Downloading weights...
Checking available files in repository...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Trying to download: models/yolo11/pt/yolo11l.pt


models/yolo11/pt/yolo11l.pt:   0%|          | 0.00/51.2M [00:00<?, ?B/s]

✅ Model downloaded successfully to /content/models/models/yolo11/pt/yolo11l.pt
Error: Could not find the video file at '/content/Test-Vids/ironsite-2.mp4'. Please upload it or fix the path.


## Train on Sample Dataset (PPE)

Use your **Sample_Dataset** (small stratified sample) to train a YOLO model so you can verify the pipeline before uploading the full 18GB.

1. Upload `Sample_Dataset` to Google Drive (e.g. `My Drive/ironsite-hackathon-project-safety_assistant/Sample_Dataset` with `images/` and `labels/` inside), **or** zip it and upload to Colab.
2. Set `SAMPLE_DATASET_PATH` below to point to it (after mounting Drive, or after unzipping to `/content`).
3. Run the cells; they will create a train/val split and `data.yaml`, then start training.

In [ ]:
# --- 1. Point to your dataset folder (choose one) ---
# Option A: From Google Drive (after running drive.mount above)
SAMPLE_DATASET_PATH = "/content/drive/MyDrive/dataset/Sample_Dataset_25k"

# Option B: Zip uploaded to Colab (avoids browser killing upload on refresh)
# !unzip -q /content/Sample_Dataset_25k.zip -d /content
# SAMPLE_DATASET_PATH = "/content/Sample_Dataset_25k"

import os
import shutil
import random
from pathlib import Path

SAMPLE_DATASET_PATH = Path(SAMPLE_DATASET_PATH)
OUTPUT_ROOT = Path("/content/PPE_Sample")
VAL_RATIO = 0.2
SEED = 42
RESUME = True   # Skip files already in OUTPUT_ROOT (re-run after interrupt to continue)

# ---------- Step 1: Path check ----------
print("[Step 1/7] Checking dataset path...")
print(f"  Path: {SAMPLE_DATASET_PATH}")
if not SAMPLE_DATASET_PATH.exists():
    print("ERROR: Path does not exist.")
    print("  • Re-run the Drive mount cell and try again.")
    print("  • Check spelling and that the folder is under /content/drive/MyDrive/...")
    try:
        drive_root = Path("/content/drive/MyDrive")
        if drive_root.exists():
            print(f"  • Folders in MyDrive: {[p.name for p in drive_root.iterdir() if p.is_dir()][:20]}")
    except Exception:
        pass
    raise FileNotFoundError(f"Dataset not found at {SAMPLE_DATASET_PATH}")

if not SAMPLE_DATASET_PATH.is_dir():
    raise FileNotFoundError(f"Path is not a directory: {SAMPLE_DATASET_PATH}")
print("  [Step 1/7] FINISHED: Dataset path exists and is a directory.")
print()

# ---------- Step 2: images/ and labels/ folders ----------
print("[Step 2/7] Checking images/ and labels/ subfolders...")
img_dir = SAMPLE_DATASET_PATH / "images"
lbl_dir = SAMPLE_DATASET_PATH / "labels"
if not img_dir.is_dir():
    print(f"ERROR: Missing 'images' folder at {img_dir}")
    print("  Your folder must contain subfolders: images/ and labels/")
    raise FileNotFoundError(f"Missing {img_dir}")
if not lbl_dir.is_dir():
    print(f"ERROR: Missing 'labels' folder at {lbl_dir}")
    raise FileNotFoundError(f"Missing {lbl_dir}")
print("  [Step 2/7] FINISHED: images/ and labels/ folders found.")
print()

# ---------- Step 3: Scan and match pairs ----------
print("[Step 3/7] Scanning for image-label pairs...")
n_images = sum(1 for f in img_dir.iterdir() if f.is_file() and f.suffix.lower() in (".jpg", ".jpeg", ".png") and not f.name.startswith("."))
n_labels = sum(1 for f in lbl_dir.iterdir() if f.is_file() and f.suffix.lower() == ".txt" and not f.name.startswith("."))
print(f"  Raw counts: {n_images} images, {n_labels} labels.")

pairs = []
for f in img_dir.iterdir():
    if not f.is_file() or f.suffix.lower() not in (".jpg", ".jpeg", ".png") or f.name.startswith("."):
        continue
    stem = f.stem
    lbl = lbl_dir / f"{stem}.txt"
    if lbl.exists():
        pairs.append((f, lbl))

orphan_images = n_images - len(pairs)
orphan_labels = n_labels - len(pairs)
if orphan_images > 0 or orphan_labels > 0:
    print(f"  → {len(pairs)} matched pairs. Orphan: {orphan_images} images without label, {orphan_labels} labels without image.")
if len(pairs) == 0:
    print("ERROR: No image–label pairs found.")
    print("  • If upload was interrupted (e.g. page refresh), re-upload or use a ZIP:")
    print("    1. Zip Sample_Dataset_25k on your PC, upload the single .zip to Drive/Colab.")
    print("    2. Unzip in Colab: !unzip -q /path/to/Sample_Dataset_25k.zip -d /content")
    print("  • Or use Google Drive for Desktop to sync the folder instead of browser upload.")
    raise FileNotFoundError("No valid image-label pairs in dataset.")

if len(pairs) < 100:
    print(f"  WARNING: Only {len(pairs)} pairs. Expected thousands for 25k dataset. Check for partial upload.")
print(f"  [Step 3/7] FINISHED: {len(pairs)} matched image-label pairs.")
print()

# ---------- Step 4: Train/val split ----------
print("[Step 4/7] Creating train/val split...")
random.seed(SEED)
random.shuffle(pairs)
n_val = max(1, int(len(pairs) * VAL_RATIO))
train_pairs, val_pairs = pairs[:-n_val], pairs[-n_val:]
print(f"  Train: {len(train_pairs)}, Val: {len(val_pairs)}.")
print("  [Step 4/7] FINISHED: Split complete.")
print()

# ---------- Step 5: Copy to OUTPUT_ROOT (with optional resume) ----------
print(f"[Step 5/7] Copying files to {OUTPUT_ROOT} (RESUME={RESUME})...")
for split, pair_list in [("train", train_pairs), ("val", val_pairs)]:
    (OUTPUT_ROOT / split / "images").mkdir(parents=True, exist_ok=True)
    (OUTPUT_ROOT / split / "labels").mkdir(parents=True, exist_ok=True)
    copied = 0
    skipped = 0
    for i, (img_path, lbl_path) in enumerate(pair_list):
        dst_img = OUTPUT_ROOT / split / "images" / img_path.name
        dst_lbl = OUTPUT_ROOT / split / "labels" / (lbl_path.stem + ".txt")
        if RESUME and dst_img.exists() and dst_lbl.exists():
            skipped += 1
        else:
            shutil.copy2(img_path, dst_img)
            shutil.copy2(lbl_path, dst_lbl)
            copied += 1
        if (i + 1) % 100 == 0 or i + 1 == len(pair_list):
            print(f"    {split}: {i+1}/{len(pair_list)} (copied {copied}, skipped {skipped})")
    if skipped > 0:
        print(f"    {split}: resumed — skipped {skipped} already present.")
    print(f"  [Step 5/7] FINISHED: {split} split copy complete (copied {copied}, skipped {skipped}).")
print()

# ---------- Step 6: Verify copy ----------
print("[Step 6/7] Verifying copied files...")
train_imgs = len(list((OUTPUT_ROOT / "train" / "images").iterdir()))
train_lbls = len(list((OUTPUT_ROOT / "train" / "labels").iterdir()))
val_imgs = len(list((OUTPUT_ROOT / "val" / "images").iterdir()))
val_lbls = len(list((OUTPUT_ROOT / "val" / "labels").iterdir()))
print(f"  train/images: {train_imgs}, train/labels: {train_lbls}")
print(f"  val/images: {val_imgs}, val/labels: {val_lbls}")
if train_imgs != train_lbls or val_imgs != val_lbls:
    print("  WARNING: Mismatch — check for incomplete copy.")
else:
    print("  Counts match.")
print("  [Step 6/7] FINISHED: Verification complete.")
print()

# ---------- Step 7: Write data.yaml ----------
print("[Step 7/7] Writing data.yaml...")
DATA_YAML = OUTPUT_ROOT / "data.yaml"
DATA_YAML.write_text(f"""# PPE Sample Dataset - 7 classes
path: {OUTPUT_ROOT}
train: train/images
val: val/images
nc: 7
names:
  0: person
  1: helmet
  2: gloves
  3: vest
  4: no helmet
  5: no gloves
  6: no vest
""")
print(f"  [Step 7/7] FINISHED: data.yaml written to {DATA_YAML}")
print()
print("=" * 60)
print("ALL STEPS FINISHED. Dataset ready for training.")
print(f"  Output root: {OUTPUT_ROOT}")
print(f"  Train: {len(train_pairs)} images | Val: {len(val_pairs)} images")
print(f"  data.yaml: {DATA_YAML}")
print("=" * 60)

[Step 1/7] Checking dataset path...
  Path: /content/drive/MyDrive/dataset/Sample_Dataset_25k
  [Step 1/7] FINISHED: Dataset path exists and is a directory.

[Step 2/7] Checking images/ and labels/ subfolders...
  [Step 2/7] FINISHED: images/ and labels/ folders found.

[Step 3/7] Scanning for image-label pairs...
  Raw counts: 24049 images, 25000 labels.
  → 24049 matched pairs. Orphan: 0 images without label, 951 labels without image.
  [Step 3/7] FINISHED: 24049 matched image-label pairs.

[Step 4/7] Creating train/val split...
  Train: 19240, Val: 4809.
  [Step 4/7] FINISHED: Split complete.

[Step 5/7] Copying files to /content/PPE_Sample (RESUME=True)...
    train: 100/19240 (copied 100, skipped 0)
    train: 200/19240 (copied 200, skipped 0)
    train: 300/19240 (copied 300, skipped 0)
    train: 400/19240 (copied 400, skipped 0)
    train: 500/19240 (copied 500, skipped 0)
    train: 600/19240 (copied 600, skipped 0)
    train: 700/19240 (copied 700, skipped 0)
    train: 800/1

In [ ]:
# Train YOLO on the sample dataset (run after the cell above)
from ultralytics import YOLO

# Start from pretrained COCO weights (yolo11n = nano, fast; yolo11s = small, better accuracy)
model = YOLO("yolo11n.pt")

results = model.train(
    data="/content/PPE_Sample/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    project="/content/runs/ppe_sample",
    name="train",
    exist_ok=True,
)

# Best weights are saved at: runs/ppe_sample/train/weights/best.pt
# To use for inference: model = YOLO("/content/runs/ppe_sample/train/weights/best.pt")

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA H100 80GB HBM3, 81079MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/PPE_Sample/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100,

RuntimeError: Dataset '/content/PPE_Sample/data.yaml' error ❌ '/content/PPE_Sample/data.yaml' does not exist

## Test your trained model on video

Run the cell below **after** training. Set `VIDEO_PATH` to your video file (e.g. on Drive or upload to Colab). Output is saved to `hazard_results/` as `{video_name}_hazards.mp4`.

In [ ]:
# Use your trained model (run after the training cell above)
TRAINED_WEIGHTS = "/content/runs/ppe_sample/train/weights/best.pt"

# Your video: set to a path on Colab (e.g. after uploading or from Drive)
VIDEO_PATH = "/content/clip_nohands.mp4"
# From Drive: VIDEO_PATH = "/content/drive/MyDrive/your_folder/your_video.mp4"

if not os.path.exists(TRAINED_WEIGHTS):
    print(f"Trained weights not found at {TRAINED_WEIGHTS}. Train the model first (run the training cell).")
elif not os.path.exists(VIDEO_PATH):
    print(f"Video not found at {VIDEO_PATH}. Upload a video or set VIDEO_PATH to your file.")
else:
    print("Running your trained PPE model on video...")
    process_video_file(
        model_path=TRAINED_WEIGHTS,
        video_path=VIDEO_PATH,
        conf_threshold=0.30,
        output_dir="hazard_results",
    )
    print("Done. Check hazard_results/ for the output video.")

Running your trained PPE model on video...
Processing 381 frames...
Processed 30/381 frames... Detections: 0, Hazards: 0
Processed 60/381 frames... Detections: 1, Hazards: 0
Processed 90/381 frames... Detections: 1, Hazards: 0
Processed 120/381 frames... Detections: 1, Hazards: 0
Processed 150/381 frames... Detections: 1, Hazards: 0
Processed 180/381 frames... Detections: 1, Hazards: 0
Processed 210/381 frames... Detections: 1, Hazards: 0
Processed 240/381 frames... Detections: 1, Hazards: 0
Processed 270/381 frames... Detections: 3, Hazards: 0
Processed 300/381 frames... Detections: 3, Hazards: 0
Processed 330/381 frames... Detections: 3, Hazards: 0
Processed 360/381 frames... Detections: 10, Hazards: 0
Done! Video saved to hazard_results/clip_nohands_hazards.mp4
Done. Check hazard_results/ for the output video.
